#  ⭐ `dim_detentor_registro`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root,add_key_value_columns,fuzzy_merge
import pandas as pd
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [2]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet")  
bronze['DETENTOR_REGISTRO']= bronze['DETENTOR_REGISTRO'].fillna('DESCONHECIDO')
bronze = (
    bronze[['DETENTOR_REGISTRO']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['DETENTOR_REGISTRO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [3]:
bronze.shape

(10940, 2)

In [4]:
bronze.DETENTOR_REGISTRO.nunique()

10940

In [5]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_DETENTOR_REGISTRO.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


### canonical

In [6]:
silver = pd.read_excel(Path(project_root) /  "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro_MANUAL.xlsx")
silver = silver.drop_duplicates(subset=['DETENTOR_REGISTRO'])
silver.head()


,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOLT,ABBOT
2,ABBOT,ABBOT
3,ABBOT BRASIL,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,ABBOT


In [7]:
silver.shape

(4078, 2)

In [8]:
silver.DETENTOR_REGISTRO.value_counts().head(10)

DETENTOR_REGISTRO
SANOFI-AVENTS\tSANOFI\n40MG SANOFI AS369\tSANOFI\nAVENTIS - SANOFI\tSANOFI\nCHATEM INC. 1715 / SANOFI AVENTIS FARMACEUTICA LTDA\tSANOFI\nGENZYME - SANOFI\tSANOFI\nGRUPO SANOFI-AVENTIS\tSANOFI\nLABORATORIO SANOF\tSANOFI\nLABORATORIO SANOFI\tSANOFI\nRIFAPENTINA (SANOFI) ISONIAZIDA (FARMANGUINHOS)\tSANOFI\nSAFONI\tSANOFI\nSAFONI AVENTIS\tSANOFI\nSAMOFI\tSANOFI\nSANAFI MEDLEY\tSANOFI\nSANFOFI\tSANOFI\nSANIFI\tSANOFI\nSANIFI AVENTIS\tSANOFI\nSANIFI MEDLEY\tSANOFI\nSANIFI MEDLEY FARMACEUTICA LTDA.\tSANOFI\nSANO\tSANOFI\nSANOBIO\tSANOFI\nSANOBIOL\tSANOFI\nSANOBIOL LTDA\tSANOFI\nSANOF\tSANOFI\nSANOFFI\tSANOFI\nSANOFFI AVENTIS\tSANOFI\nSANOFI\tSANOFI\nSANOFI - A\tSANOFI\nSANOFI - AS369\tSANOFI\nSANOFI - AVENTIS\tSANOFI\nSANOFI - LOTE: AS541\tSANOFI\nSANOFI - LOTE:AS770A\tSANOFI\nSANOFI - TEVA\tSANOFI\nSANOFI -AVENTIS\tSANOFI\nSANOFI -AVENTIS DE MEXICO S.A DE C.V\tSANOFI\nSANOFI AVENTI\tSANOFI\nSANOFI AVENTIS\tSANOFI\nSANOFI AVENTIS -\tSANOFI\nSANOFI AVENTIS 9S556\tSANOFI\nSANO

In [9]:
silver.query("DETENTOR_REGISTRO == 'BIONTECH - PFIZER'")

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO
3590,BIONTECH - PFIZER,PFIZER


In [10]:
hist = add_key_value_columns(
    silver, #.query("DETENTOR_REGISTRO_PADRONIZADO != 'DESCONHECIDO'"),
    base_col="DETENTOR_REGISTRO",
    normalized_col="DETENTOR_REGISTRO_PADRONIZADO",
)

hist.head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_VALOR,DETENTOR_REGISTRO_CHAVE
0,ABACUS MEDICINE,ABACUS MEDICINE,ABACUS MEDICINE,1
1,ABBOLT,ABBOT,ABBOT,2
2,ABBOT,ABBOT,ABBOT,2
3,ABBOT BRASIL,ABBOT,ABBOT,2
4,ABBOT L: 0071/20 V: 31/01/2022,ABBOT,ABBOT,2


In [11]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet", index=False)

# 🥇 Gold


In [12]:
hist.columns

Index(['DETENTOR_REGISTRO', 'DETENTOR_REGISTRO_PADRONIZADO',
       'DETENTOR_REGISTRO_VALOR', 'DETENTOR_REGISTRO_CHAVE'],
      dtype='object')

In [13]:
dim = hist[['DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates() 

In [14]:

dim.to_csv(Path(project_root) / "data/03_gold/dim_detentor_registro/dim_detentor_registro.csv", sep=";", index=False)